In [7]:
import numpy as np
import matplotlib.pyplot as plt

In [8]:
class NArmedBanditAgent():
    def __init__(self, n_arms = 10, epsilon = 0.1, decay_rate = 0):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.decay_rate = decay_rate
        self.reset()
    
    def __str__(self):
        if self.epsilon == 0:
            return "Greedy"
        else:
            return "Epsilon = " + str(self.epsilon)
    
    def epsilon_decay(self):
        self.epsilon = max(0.01, self.epsilon * (1 - self.decay_rate))


    def select_action(self):
        self.epsilon_decay()
        
        choice = np.random.choice(["explore", "exploit"], p=[self.epsilon, 1 - self.epsilon])
        
        if choice == "explore":
            return np.random.randint(0, self.n_arms - 1)
        else:
            max_indices = np.where(self.q_values == self.q_values.max())[0]
            return np.random.choice(max_indices)
    
    def q_value(self, action):
        # q(a) is the average value of rewards for the action a
        total_rewards = self.total_rewards[action]
        n_actions = self.action_count[action]

        if n_actions == 0:
            return 0
        else:
            return total_rewards/n_actions

    
    def reward_signal(self, action, reward):
        self.total_rewards[action] += reward
        self.action_count[action] += 1
        # q value is the average rewards received for an action
        self.q_values[action] = self.total_rewards[action] / self.action_count[action]
    
    def reset(self):
        self.total_rewards = np.zeros(self.n_arms)
        self.action_count = np.zeros(self.n_arms)
        self.q_values = np.zeros(self.n_arms)
        if self.decay_rate > 0:
            self.epsilon = 1.0

In [9]:
class NArmedBanditEnv():
    def __init__(self, n_actions = 10):
        self.n_actions = n_actions
        self.action_values = None
        self.optimal_action = None
        self.reset()
        
    def apply_action(self, action):
        # a random reward with mean = self.action_values[action]
        return np.random.normal(loc=self.action_values[action], scale=1)
    
    def reset(self):
        # mean = 0, standard deviation = 1         
        self.action_values = np.random.normal(0, scale=1, size = self.n_actions)
        self.optimal_action = np.argmax(self.action_values)
        

In [ ]:
agents = [NArmedBanditAgent(epsilon=0), NArmedBanditAgent(epsilon=0.01), NArmedBanditAgent(epsilon=0.1), NArmedBanditAgent(epsilon=0.5)]
# agents = [NArmedBanditAgent(epsilon=1.0, decay_rate=0.01)]
environment = NArmedBanditEnv()

n_timesteps = 1000
n_iterations = 2000

score_array = np.zeros((n_timesteps, len(agents)))
optim_array = np.zeros((n_timesteps, len(agents)))

for it in range(n_iterations):
    for agent in agents:
        agent.reset()
        environment.reset()
    
    for step in range(n_timesteps):
        for agent_id, agent in enumerate(agents):                
            action = agent.select_action()
            reward = environment.apply_action(action)
            agent.reward_signal(action, reward)
            score_array[step, agent_id] += reward

            if action == environment.optimal_action:
                optim_array[step, agent_id] += 1


for step in range(n_timesteps):
    for agent_id, agent in enumerate(agents):
        score_array[step, agent_id] /= n_iterations
        optim_array[step, agent_id] /= n_iterations

In [ ]:
plt.title("10-Armed TestBed - Average rewards")
plt.ylabel("Average rewards")
plt.xlabel("Timesteps")
plt.plot(score_array)
plt.legend(agents, loc=4)
plt.show()

In [ ]:
plt.title("10-Armed TestBed - % Optimal Action")
plt.plot(optim_array * 100)
plt.ylim(0, 100)
plt.ylabel('% Optimal Action')
plt.xlabel('Timesteps')
plt.legend(agents, loc=4)
plt.show()
